In [ ]:
from skipalignments import *
############### ENTER THE LOG PATHS HERE ###############
path_to_road_fines_log = '.../path/to/xes'
path_to_request_for_payment_log = '.../path/to/xes'
path_to_international_declarations_log = '.../path/to/xes'
inspected_log = Logs.ROAD_FINES

############### ENTER THE STOCHASTIC ESTIMATOR HERE ###############
ebi_method = EbiWeights.OCCURANCE

In [ ]:
%load_ext autoreload
%autoreload 2
import pm4py
import statistics
import random

In [ ]:
def update_pair_taus(tree:ProcessTree):
    if isinstance(tree, Tau):
        if tree.parent is not None and len(tree.parent.children) == 2:
            other = tree.parent.children[0]
            if other == tree:
                other = tree.parent.children[1]
            if isinstance(other, Activity):
                # set tau
                tree.name = "TAU_" + other.name
            else:
                tree.name = "TAU_" + other.id
        else:
            tree.name = "TAU_" + str(tree.get_distance_to_root()) + str(random.random())
        return
    elif not isinstance(tree, Activity):
        for c in tree.children:
            update_pair_taus(c)
        return
    else:
        return

In [ ]:
path = None
if inspected_log == Logs.ROAD_FINES:
    path = path_to_road_fines_log
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    path = path_to_request_for_payment_log
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    path = path_to_international_declarations_log
log_rf = pm4py.read_xes(path)

In [ ]:
threshold = None
if inspected_log == Logs.ROAD_FINES:
    threshold = 0.5
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    threshold = 0.5
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    threshold = 0.5
process_tree_rf = pm4py.discover_process_tree_inductive(log_rf, activity_key='concept:name', case_id_key='case:concept:name', timestamp_key='time:timestamp', noise_threshold=threshold)
pm4py.view_process_tree(process_tree_rf, format='png')

In [ ]:
tree_rf = ProcessTree.from_pm4py(process_tree_rf, 100000, 0, 0)
update_pair_taus(tree_rf)

In [ ]:
output_path = None
if inspected_log == Logs.ROAD_FINES:
    output_path = "./im_results/rf"
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    output_path = "./im_results/payment"
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    output_path = "./im_results/declarations"

if ebi_method == EbiWeights.OCCURANCE:
    output_path += "/occurance"
elif ebi_method == EbiWeights.UNIFORM:
    output_path += "/uniform"

In [ ]:
output_path = None
if inspected_log == Logs.ROAD_FINES:
    output_path = "./results/im/rf"
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    output_path = "./results/im/payment"
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    output_path = "./results/im/declarations"

if ebi_method == EbiWeights.OCCURANCE:
    output_path += "/occurance"
elif ebi_method == EbiWeights.UNIFORM:
    output_path += "/uniform"

In [ ]:
derivation = DerivationPipeline(tree_rf, log_rf, pn_log=log_rf, pn_method=ebi_method, sagn_timeout=600)

In [ ]:
derivation.compute(output_path)

In [ ]:
print(derivation.print_blinded())

In [ ]:
derivation.stats()